# 03 — Development Finance LGD Model

**Australian Bank Practice — Scenario-Based Approach**

Development finance is the most complex and cyclically sensitive lending product.
This notebook demonstrates:

| Step | Description |
|------|-------------|
| 1 | Load synthetic development finance default data |
| 2 | EDA — LGD by completion stage, development type, pre-sales |
| 3 | Completion-stage segmentation (THE primary LGD driver) |
| 4 | Fund-to-complete vs sell-as-is decision analysis |
| 5 | Cost-to-complete as dominant cost |
| 6 | Scenario analysis — GRV decline, cost overrun, pre-sale rescission |
| 7 | Downturn overlay by completion stage |
| 8 | MoC (highest of all products: +5pp) |
| 9 | APRA slotting framework |
| 10 | Validation |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_development_data
from src.lgd_calculation import DevelopmentLGDEngine, exposure_weighted_average
from src.validation import (
    accuracy_metrics, conservatism_test, calibration_by_segment,
    compute_psi, generate_validation_report
)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.join('..', 'outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)

## 1. Generate & Load Data

In [ ]:
loans, cashflows = generate_development_data(n_loans=200, seed=44)

print(f"Loans: {loans.shape}")
print(f"Cashflows: {cashflows.shape}")
print(f"\nCompletion stage distribution:")
display(loans['completion_stage'].value_counts())
print(f"\nFund-to-complete rate: {loans['fund_to_complete'].mean():.1%}")
loans.head()

## 2. Exploratory Data Analysis

In [ ]:
print("=== Realised LGD Summary ===")
display(loans['realised_lgd'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(loans['realised_lgd'], bins=40, edgecolor='white', alpha=0.8, color='darkorange')
axes[0].axvline(loans['realised_lgd'].mean(), color='red', ls='--',
                label=f"Mean: {loans['realised_lgd'].mean():.2%}")
axes[0].set_title('Development Finance — Realised LGD Distribution')
axes[0].set_xlabel('LGD')
axes[0].legend()

# Completion stage is THE key driver
stage_order = ['Pre-Construction', 'Early Construction', 'Mid-Construction', 'Near-Complete', 'Complete Unsold']
loans['completion_stage'] = pd.Categorical(loans['completion_stage'], categories=stage_order, ordered=True)
loans.boxplot(column='realised_lgd', by='completion_stage', ax=axes[1], rot=20)
axes[1].set_title('Realised LGD by Completion Stage')
axes[1].set_xlabel('Completion Stage at Default')
axes[1].set_ylabel('Realised LGD')
plt.suptitle('')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'development_lgd_overview.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Key driver analysis
print("=== Average LGD by Key Segments ===")
for col in ['completion_stage', 'development_type', 'default_trigger', 'fund_to_complete']:
    print(f"\n--- {col} ---")
    summary = loans.groupby(col).agg(
        count=('realised_lgd', 'size'),
        mean_lgd=('realised_lgd', 'mean'),
        median_lgd=('realised_lgd', 'median'),
        mean_ead=('ead', 'mean'),
        mean_completion=('completion_pct', 'mean'),
    ).round(4)
    display(summary)

In [ ]:
# Completion % vs LGD — coloured by fund-to-complete decision
fig, ax = plt.subplots(figsize=(10, 6))
for ftc, colour, label in [(1, 'green', 'Fund to Complete'), (0, 'salmon', 'Sell As-Is')]:
    mask = loans['fund_to_complete'] == ftc
    ax.scatter(loans.loc[mask, 'completion_pct'], loans.loc[mask, 'realised_lgd'],
              alpha=0.5, s=25, color=colour, label=label)
ax.set_xlabel('Completion % at Default')
ax.set_ylabel('Realised LGD')
ax.set_title('Completion Stage vs LGD — Fund-to-Complete Decision')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'development_completion_vs_lgd.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Completion-Stage Segmentation

**Completion stage at default is the single most important LGD driver** in development finance.
```
Level 1: Completion stage (5 bands)
  Level 2: Development type
    Level 3: Pre-sale coverage band
```

In [ ]:
engine = DevelopmentLGDEngine(moc=0.05)

# Primary segmentation
lr_lgd = engine.compute_long_run_lgd(loans, segments=['completion_stage'])
print("=== Exposure-Weighted Long-Run LGD by Completion Stage ===")
display(lr_lgd)

# With pre-sale band
segmented = engine.segment_loans(loans)
lr_lgd_detail = exposure_weighted_average(
    segmented, 'realised_lgd', 'ead',
    group_col=['completion_stage', 'presale_band']
)
print("\n=== Completion Stage × Pre-sale Band ===")
display(lr_lgd_detail)

## 4. Fund-to-Complete vs Sell-As-Is

In [ ]:
# Compare outcomes: fund-to-complete vs sell-as-is
ftc_analysis = loans.groupby(['completion_stage', 'fund_to_complete']).agg(
    count=('realised_lgd', 'size'),
    mean_lgd=('realised_lgd', 'mean'),
    mean_ead=('ead', 'mean'),
    mean_ctc=('cost_to_complete', 'mean'),
).round(4)

print("=== Fund-to-Complete Analysis ===")
print("(fund_to_complete: 0=Sell As-Is, 1=Fund to Complete)")
display(ftc_analysis)

## 5. Cost-to-Complete Analysis

In [ ]:
# Cost breakdown from cashflows
cost_cfs = cashflows[cashflows['cashflow_category'] == 'Cost']
cost_breakdown = cost_cfs.groupby('cashflow_type').agg(
    count=('amount', 'size'),
    total_amount=('amount', 'sum'),
    total_pv=('amount_pv', 'sum'),
).sort_values('total_pv', ascending=False)

cost_breakdown['pct_of_total'] = cost_breakdown['total_pv'] / cost_breakdown['total_pv'].sum()

print("=== Cost Breakdown (PV) ===")
display(cost_breakdown.round(2))

# Visualise
fig, ax = plt.subplots(figsize=(10, 5))
cost_breakdown['total_pv'].plot.barh(ax=ax, color='coral', edgecolor='white')
ax.set_xlabel('Total PV of Costs ($)')
ax.set_title('Development Finance — Cost Composition')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'development_cost_breakdown.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n→ Cost-to-complete dominates total workout costs in development finance.")

## 6. Scenario Analysis

Test LGD sensitivity to key stress parameters:
- **GRV decline**: property market downturn
- **Cost overrun**: construction cost inflation
- **Pre-sale rescission**: sunset clause exercise
- **Sales extension**: longer time to sell in weak market

In [ ]:
# Define scenarios
scenarios = {
    'Base Case': dict(grv_decline=0.0, cost_overrun=0.0, rescission_rate=0.0, sales_extension_months=0),
    'Mild Stress': dict(grv_decline=-0.10, cost_overrun=0.05, rescission_rate=0.10, sales_extension_months=3),
    'Moderate Downturn': dict(grv_decline=-0.20, cost_overrun=0.10, rescission_rate=0.20, sales_extension_months=6),
    'Severe Downturn': dict(grv_decline=-0.30, cost_overrun=0.15, rescission_rate=0.30, sales_extension_months=9),
    'Extreme Stress': dict(grv_decline=-0.40, cost_overrun=0.20, rescission_rate=0.40, sales_extension_months=12),
}

scenario_results = []
for name, params in scenarios.items():
    s = engine.scenario_analysis(loans, **params)
    mean_lgd = s['scenario_lgd'].mean()
    ewa_lgd = exposure_weighted_average(s.assign(realised_lgd=s['scenario_lgd']), 'realised_lgd')
    scenario_results.append({
        'Scenario': name,
        'Mean LGD': mean_lgd,
        'EWA LGD': ewa_lgd,
        'P95 LGD': s['scenario_lgd'].quantile(0.95),
        **params,
    })

scenario_df = pd.DataFrame(scenario_results)
print("=== Scenario Analysis Results ===")
display(scenario_df.round(4))

In [ ]:
# Visualise scenario impact
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c', '#8e44ad']
bars = ax.bar(scenario_df['Scenario'], scenario_df['EWA LGD'], color=colors, edgecolor='white')
for bar, val in zip(bars, scenario_df['EWA LGD']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Exposure-Weighted LGD')
ax.set_title('Development Finance — Scenario Analysis')
ax.set_ylim(0, scenario_df['EWA LGD'].max() * 1.25)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'development_scenario_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7–8. Regulatory Pipeline: Downturn + MoC

In [ ]:
result = engine.apply_overlays(loans)

pipeline_cols = ['realised_lgd', 'lgd_downturn', 'lgd_with_moc', 'lgd_final']

print("=== Regulatory Pipeline ===")
display(result[pipeline_cols].describe().round(4))

print("\n=== Exposure-Weighted by Completion Stage ===")
for stage in stage_order:
    sub = result[result['completion_stage'] == stage]
    if len(sub) == 0:
        continue
    ewa = exposure_weighted_average(sub, 'lgd_final')
    scalar = sub['downturn_scalar'].iloc[0]
    print(f"  {stage:25s}: Final LGD={ewa:.2%}  (scalar={scalar:.2f}, +MoC {engine.moc:.0%})  n={len(sub)}")

## 9. APRA Slotting Framework

APRA may classify development finance as **specialised lending**, requiring a slotting approach:

| Slot | Risk Weight |
|------|-------------|
| Strong | 70% |
| Good | 90% |
| Satisfactory | 115% |
| Weak | 250% |

In [ ]:
slotted = engine.assign_slotting(result)

slot_summary = slotted.groupby('slotting_category').agg(
    count=('loan_id', 'size'),
    mean_lgd=('lgd_final', 'mean'),
    mean_rw=('slotting_rw', 'mean'),
    total_ead=('ead', 'sum'),
    mean_completion=('completion_pct', 'mean'),
    mean_presales=('presale_coverage', 'mean'),
).round(4)

print("=== APRA Slotting Summary ===")
display(slot_summary)

# Illustrative slotted RWA
slotted['slotted_rwa'] = slotted['ead'] * slotted['slotting_rw']
print(f"\nTotal Slotted RWA: ${slotted['slotted_rwa'].sum():,.0f}")
print(f"Average Risk Weight: {slotted['slotted_rwa'].sum() / slotted['ead'].sum():.2%}")

In [ ]:
# Slotting distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

slot_counts = slotted['slotting_category'].value_counts().reindex(['Strong', 'Good', 'Satisfactory', 'Weak'])
slot_colours = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
axes[0].bar(slot_counts.index, slot_counts.values, color=slot_colours, edgecolor='white')
axes[0].set_title('Slotting Category Distribution')
axes[0].set_ylabel('Number of Loans')

slotted.boxplot(column='lgd_final', by='slotting_category', ax=axes[1])
axes[1].set_title('Final LGD by Slotting Category')
axes[1].set_ylabel('Final LGD')
plt.suptitle('')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'development_slotting.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Validation

In [ ]:
val_report = generate_validation_report(
    result,
    actual_col='realised_lgd',
    predicted_col='lgd_final',
    segment_col='completion_stage',
    date_col='default_date'
)

print("=== Accuracy ===")
for k, v in val_report['accuracy'].items():
    print(f"  {k}: {v}")

print("\n=== Conservatism Test ===")
for k, v in val_report['conservatism'].items():
    print(f"  {k}: {v}")

print("\n=== Calibration by Completion Stage ===")
display(val_report.get('calibration', 'N/A'))

In [ ]:
# Save outputs
slotted.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'development_lgd_results.csv'), index=False)
lr_lgd.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'development_segment_summary.csv'), index=False)
scenario_df.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'development_scenario_analysis.csv'), index=False)

print("Outputs saved to outputs/tables/")